In [1]:
# Test script 4

In [49]:
import os
import xarray as xr
import numpy as np
import dask.array as da
from utils.mortality_utils import att_frac
from utils.mortality_utils import mortality

In [50]:
# === Path config ===
BMR_DIR = "/glade/work/awells/air_quality/BMR/"
POP_DIR = "/glade/work/awells/air_quality/SSP_pop/SSP2/"
O3_DIR = "/glade/work/awells/air_quality/CESM/ozone/OSDMA8_BC/"

In [51]:
# === Load data ===
bmr_file = "GBD_BMR_Country_Mask_COPD_1990-2009.nc"
bmr_path = os.path.join(BMR_DIR, bmr_file)
BMR = xr.open_dataarray(bmr_path)  # three quantiles

pop_file = "ssp2_coarse_grid_annual_2000-2100.nc"
pop_path = os.path.join(POP_DIR, pop_file)
population = xr.open_dataarray(pop_path)

In [52]:
n_samples = 1000

# === Scalar distributions (assuming normal dist.) ===
# TMREL from GBD21
tmrel_mean = 32.4
tmrel_std = (35.7 - 29.1) / (2 * 1.96)
tmrel_samples = np.random.normal(tmrel_mean, tmrel_std, size=n_samples)

# Beta from RR per 10ppb
RR_10 = 1.074
RR_10_lower = 1.014
RR_10_upper = 1.137
beta_mean = np.log(RR_10) / 10
beta_std = (np.log(RR_10_upper) - np.log(RR_10_lower)) / (2 * 1.96 * 10)
beta_samples = np.random.normal(beta_mean, beta_std, size=n_samples)

# Convert to Dask for broadcasting
tmrel_dask = da.from_array(tmrel_samples[:, np.newaxis, np.newaxis],
                           chunks=(100, 1, 1))
beta_dask = da.from_array(beta_samples[:, np.newaxis, np.newaxis],
                          chunks=(100, 1, 1))

In [53]:
def monte_carlo(n_samples, BMR, POP, O3):
    lat = BMR["lat"]
    lon = BMR["lon"]

    # === BMR ===
    bmr_m = BMR.sel(quantile="mean")
    bmr_l = BMR.sel(quantile="lower")
    bmr_u = BMR.sel(quantile="upper")

    BMR_mean = bmr_m.chunk({"lat": 180, "lon": 360})
    BMR_lower = bmr_l.chunk({"lat": 180, "lon": 360})
    BMR_upper = bmr_u.chunk({"lat": 180, "lon": 360})

    # Standard deviation for BMR
    bmr_std = (BMR_upper - BMR_lower) / (2 * 1.96)

    # Sample BMR: shape = (samples, lat, lon)
    bmr_samples = da.random.normal(
        loc=BMR_mean.data, scale=bmr_std.data,
        size=(n_samples, len(lat), len(lon)),
        chunks=(100, 180, 360))

    POP = POP.chunk({"lat": 180, "lon": 360})
    O3 = O3.chunk({"lat": 180, "lon": 360})

    # Broadcast POP and x to sample dimension
    pop_samples = da.broadcast_to(POP.data, (n_samples, len(lat), len(lon)))
    O3_samples = da.broadcast_to(O3.data, (n_samples, len(lat), len(lon)))

    return bmr_samples, pop_samples, O3_samples

In [54]:
# === Path config ===
SAVE_DIR = "/glade/work/awells/air_quality/CESM/mortality/"
SCENARIOS = ["ARISE", "SSP245"]

# === Main loop ===
for scenario in SCENARIOS:
    for ens_num in range(1, 2):
        print(f"Processing {scenario}, Ensemble {ens_num:02d}")
        if scenario == "ARISE":
            dates = "2035-2068"
        elif scenario == "SSP245":
            dates = "2020-2068"

        o3_file = f"OSDMA8_BC_CESM2_{scenario}_{ens_num:02d}_{dates}.nc"
        o3_path = os.path.join(O3_DIR, o3_file)
        o3 = xr.open_dataarray(o3_path)

        # Adjust indices to match (with small tolerance)
        # e.g., max 1e-7 km distance
        o3 = o3.reindex_like(BMR, method="nearest", tolerance=1e-9, fill_value=0)
        population = population.reindex_like(BMR, method="nearest", tolerance=1e-9)

        # Flag if any nans present (i.e. reindex was out of tolerance distance)
        assert not population.isnull().any()

        M_mean = []
        M_lower = []
        M_upper = []

        for year in o3["year"].values:
            print(f"Processing year {year}")
            POP = population.sel(year=year)
            O3 = o3.sel(year=year)
            BMR_samples, POP_samples, O3_samples = monte_carlo(n_samples, BMR,
                                                               POP, O3)

            AF = att_frac(O3_samples, tmrel_dask, beta_dask)
            mortality_samples = mortality(AF, BMR_samples, POP_samples)
            mortality_year = xr.DataArray(
                mortality_samples,
                dims=("sample", "lat", "lon"),
                coords={"sample": np.arange(n_samples),
                        "lat": BMR.lat, "lon": BMR.lon},
                name="Mortality"
            ).chunk({"sample": -1})

            M_mean.append(mortality_year.mean("sample"))
            M_lower.append(mortality_year.quantile(0.025, dim="sample"))
            M_upper.append(mortality_year.quantile(0.975, dim="sample"))

        mean_timeseries = xr.concat(
            M_mean,
            dim=(xr.DataArray(o3["year"].values,
                              dims="year", name="year"))
        )
        lower_timeseries = xr.concat(
            M_lower,
            dim=(xr.DataArray(o3["year"].values,
                              dims="year", name="year"))
        ).drop_vars("quantile")
        upper_timeseries = xr.concat(
            M_upper,
            dim=(xr.DataArray(o3["year"].values,
                              dims="year", name="year"))
        ).drop_vars("quantile")

        ds = xr.Dataset({
            "Mortality_mean": mean_timeseries,
            "Mortality_2.5pct": lower_timeseries,
            "Mortality_97.5pct": upper_timeseries,
        })

        print("Loading data to memory")
        ds_out = ds.compute()

        out_file = f"TESTMortality_CESM2_{scenario}_{ens_num:02d}_{dates}.nc"
        out_path = os.path.join(SAVE_DIR, out_file)

        print(f"Saving mortality timeseries to {out_path}")
        description = ("Total COPD mortality due to surface ozone - scripts "
                       "by A.F. Wells (2025)")
        ds.attrs["description"] = description
        ds.attrs["ensemble_number"] = ens_num
        ds.attrs["scenario"] = scenario
        ds.to_netcdf(out_path)

Processing ARISE, Ensemble 01
Processing year 2035
Processing year 2036


/glade/work/awells/conda-envs/my-env/lib/python3.12/site-packages/dask/array/core.py:4867: PerformanceWarning: Increasing number of chunks by factor of 10
  result = blockwise(
/glade/work/awells/conda-envs/my-env/lib/python3.12/site-packages/dask/array/core.py:4867: PerformanceWarning: Increasing number of chunks by factor of 10
  result = blockwise(


Processing year 2037
Processing year 2038


/glade/work/awells/conda-envs/my-env/lib/python3.12/site-packages/dask/array/core.py:4867: PerformanceWarning: Increasing number of chunks by factor of 10
  result = blockwise(
/glade/work/awells/conda-envs/my-env/lib/python3.12/site-packages/dask/array/core.py:4867: PerformanceWarning: Increasing number of chunks by factor of 10
  result = blockwise(


Processing year 2039
Processing year 2040


...

/glade/work/awells/conda-envs/my-env/lib/python3.12/site-packages/dask/array/core.py:4867: PerformanceWarning: Increasing number of chunks by factor of 10
  result = blockwise(
/glade/work/awells/conda-envs/my-env/lib/python3.12/site-packages/dask/array/core.py:4867: PerformanceWarning: Increasing number of chunks by factor of 10
  result = blockwise(


Processing year 2067
Processing year 2068


/glade/work/awells/conda-envs/my-env/lib/python3.12/site-packages/dask/array/core.py:4867: PerformanceWarning: Increasing number of chunks by factor of 10
  result = blockwise(
/glade/work/awells/conda-envs/my-env/lib/python3.12/site-packages/dask/array/core.py:4867: PerformanceWarning: Increasing number of chunks by factor of 10
  result = blockwise(


Loading data to memory



KeyboardInterrupt



In [65]:
mortality_year

<xarray.DataArray 'Mortality' (sample: 1000, lat: 1800, lon: 3600)> Size: 52GB
dask.array<rechunk-merge, shape=(1000, 1800, 3600), dtype=float64, chunksize=(1000, 180, 360), chunktype=numpy.ndarray>
Coordinates:
  * sample   (sample) int64 8kB 0 1 2 3 4 5 6 7 ... 993 994 995 996 997 998 999
  * lat      (lat) float64 14kB -89.95 -89.85 -89.75 ... 89.75 89.85 89.95
  * lon      (lon) float64 29kB -179.9 -179.8 -179.8 ... 179.8 179.8 179.9

In [62]:
mortality_year.sum(dim=("lat", "lon")).quantile(0.025, dim="sample").values

array(102108.69684945)

In [63]:
mortality_year.sum(dim=("lat", "lon")).quantile(0.025, dim="sample").compute()

<xarray.DataArray 'Mortality' ()> Size: 8B
array(102108.69684945)
Coordinates:
    quantile  float64 8B 0.025

In [64]:
mortality_year.sum(dim=("lat", "lon")).mean("sample").compute()

<xarray.DataArray 'Mortality' ()> Size: 8B
array(417725.22637101)

In [9]:
m = mortality_year.persist()

In [38]:
m = mortality_year.chunk({"sample": -1})

In [39]:
m_mean = m.mean("sample")
m_lower = m.quantile(0.025, dim="sample")
m_upper = m.quantile(0.975, dim="sample")

In [40]:
lower = m_lower.drop_vars("quantile")
upper = m_upper.drop_vars("quantile")

In [41]:
ds_out = xr.Dataset({
    "M_mean": m_mean,
    "M_2.5pct": lower,
    "M_97.5pct": upper,
})

In [43]:
ds_out

<xarray.Dataset> Size: 156MB
Dimensions:    (lat: 1800, lon: 3600)
Coordinates:
  * lat        (lat) float64 14kB -89.95 -89.85 -89.75 ... 89.75 89.85 89.95
  * lon        (lon) float64 29kB -179.9 -179.8 -179.8 ... 179.8 179.8 179.9
Data variables:
    M_mean     (lat, lon) float64 52MB dask.array<chunksize=(180, 360), meta=np.ndarray>
    M_2.5pct   (lat, lon) float64 52MB dask.array<chunksize=(180, 360), meta=np.ndarray>
    M_97.5pct  (lat, lon) float64 52MB dask.array<chunksize=(180, 360), meta=np.ndarray>

In [44]:
ds = ds_out.compute()

In [47]:
print(f"{ds.nbytes / 1e6:.2f} MB")
print(f"{ds.nbytes / 1e9:.2f} GB")

155.56 MB
0.16 GB
